In [ ]:
import sys
import subprocess

paquetes = [
    'scikit-image',
    'rasterio',
    'geopandas',
    'scipy',
    'numpy',
    'pandas',
    'pathlib',
    'affine'
]

for paquete in paquetes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', paquete], 
                   capture_output=True)
    print(f"✓ {paquete}")

print("\nTodo instalado")

## Versión Anterior

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.transform import from_bounds
import numpy as np
import pandas as pd
from scipy.ndimage import distance_transform_edt
from pathlib import Path
import os
import json

In [ ]:
# ============================================================
# RUTAS — ajusta según tu estructura
# ============================================================
PATH_MUNICIPIOS  = "../data/Municipios_Diciembre_2025_shp/Munpio.shp"
PATH_NDVI        = "../data/landsat/landsat_ndvi/"        # archivos ndvi_XXXXX.tif
PATH_DEM         = "../data/dem/dem_srtm/"       # archivos dem_XXXXX.tif
PATH_HIDRO       = "../data/hidrografia/"    # archivos XXXXX_municipio.gpkg
PATH_VIAS        = "../data/red_vial/RedVialOD/RedVial.shp"
PATH_POBLADOS    = "../data/centros_poblados/"
PATH_DILIGENCIAS = "../data/Processed/full_MIG.geojson"
PATH_OUTPUT      = "../data/Processed/"

os.makedirs(PATH_OUTPUT, exist_ok=True)

RESOLUCION = 30  # metros
CRS = "EPSG:4326"


In [ ]:
from pathlib import Path

archivos_osm = list(Path("../data/red_vial_osm/").glob("*.gpkg"))
print(f"Archivos OSM: {len(archivos_osm)}")
for f in sorted(archivos_osm)[:5]:
    print(f"  {f.name}")

In [ ]:

# ============================================================
# CARGAR DATOS BASE
# ============================================================
print("Cargando datos base...")

municipios = gpd.read_file(PATH_MUNICIPIOS)
municipios['MpCodigo'] = municipios['MpCodigo'].astype(str).str.zfill(5)
municipios = municipios.to_crs(CRS)

vias = gpd.read_file(PATH_VIAS).to_crs(CRS)

with open(PATH_DILIGENCIAS) as f:
    dilig = json.load(f)
props = [feat['properties'] for feat in dilig['features']]
coords = [feat['geometry']['coordinates'] for feat in dilig['features']]
df_dilig = pd.DataFrame(props)
df_dilig['lon'] = [c[0] for c in coords]
df_dilig['lat'] = [c[1] for c in coords]
df_dilig['Cod_Mun'] = df_dilig['Cod_Mun'].astype(str).str.zfill(5)

gdf_dilig = gpd.GeoDataFrame(
    df_dilig,
    geometry=gpd.points_from_xy(df_dilig['lon'], df_dilig['lat']),
    crs=CRS
)

print(f"  ✓ {len(municipios)} municipios")
print(f"  ✓ {len(gdf_dilig)} diligencias")



In [ ]:
# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def raster_distancia(gdf_lineas, transform, shape, crs):
    """Convierte GeoDataFrame a raster de distancia euclidiana."""
    from rasterio.features import rasterize
    from affine import Affine
    
    burned = rasterize(
        [(geom, 1) for geom in gdf_lineas.geometry if geom is not None],
        out_shape=shape,
        transform=transform,
        fill=0,
        dtype='uint8'
    )
    # Distancia en píxeles → convertir a metros aproximados
    dist = distance_transform_edt(1 - burned) * RESOLUCION
    return dist

def derivar_pendiente(dem_array):
    """Calcula pendiente en grados a partir del DEM."""
    dy, dx = np.gradient(dem_array, RESOLUCION)
    pendiente = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))
    return pendiente

def cargar_raster_municipio(path_carpeta, codigo, prefijo):
    """Carga raster de un municipio específico."""
    # Buscar archivo con el código
    carpeta = Path(path_carpeta)
    archivos = list(carpeta.glob(f"*{codigo}*.tif"))
    if not archivos:
        return None, None
    with rasterio.open(archivos[0]) as src:
        arr = src.read(1).astype(float)
        arr[arr == src.nodata] = np.nan
        return arr, src.transform



In [ ]:
excluir_tipos = ['Cementerio']

In [ ]:
# ============================================================
# PROCESAR CADA MUNICIPIO
# ============================================================
codigos = df_dilig['Cod_Mun'].unique()
codigos = [c for c in codigos if c != '000SD']

resultados = []
errores = []

for codigo in codigos:
    print(f"\nProcesando: {codigo}")
    
    try:
        # Geometría del municipio
        mun = municipios[municipios['MpCodigo'] == codigo]
        if len(mun) == 0:
            print(f"  ✗ Municipio no encontrado en shapefile")
            errores.append((codigo, 'municipio no encontrado'))
            continue
        
        geom = mun.geometry.values[0]
        bounds = mun.total_bounds  # minx, miny, maxx, maxy
        
        # Definir grid de referencia
        width  = max(int((bounds[2] - bounds[0]) / (RESOLUCION / 111320)), 10)
        height = max(int((bounds[3] - bounds[1]) / (RESOLUCION / 111320)), 10)
        transform = from_bounds(*bounds, width, height)
        shape = (height, width)

        # ── 1. NDVI ──────────────────────────────────────────
        ndvi_arr, _ = cargar_raster_municipio(PATH_NDVI, codigo, 'ndvi')
        if ndvi_arr is None:
            print(f"  ✗ NDVI no encontrado")
            errores.append((codigo, 'NDVI no encontrado'))
            continue
        # Redimensionar al grid de referencia
        from skimage.transform import resize
        ndvi_arr = resize(ndvi_arr, shape, anti_aliasing=True)
        print(f"  ✓ NDVI: {ndvi_arr.shape}")

        # ── 2. DEM → altitud y pendiente ────────────────────
        dem_arr, _ = cargar_raster_municipio(PATH_DEM, codigo, 'dem')
        if dem_arr is None:
            print(f"  ✗ DEM no encontrado")
            errores.append((codigo, 'DEM no encontrado'))
            continue
        dem_arr   = resize(dem_arr, shape, anti_aliasing=True)
        slope_arr = derivar_pendiente(dem_arr)
        print(f"  ✓ DEM + pendiente: {dem_arr.shape}")

        # ── 3. Distancia a hidrografía ───────────────────────
        hidro_path = Path(PATH_HIDRO)
        hidro_files = list(hidro_path.glob(f"{codigo}_*.gpkg"))
        if hidro_files:
            hidro = gpd.read_file(hidro_files[0]).to_crs(CRS)
            dist_hidro = raster_distancia(hidro, transform, shape, CRS)
        else:
            dist_hidro = np.full(shape, np.nan)
            print(f"  ! Hidrografía no encontrada — usando NaN")
        print(f"  ✓ Distancia hidrografía")

        # ── 4. Distancia a vías ──────────────────────────────
        vias_clip = vias.clip(mun)
        if len(vias_clip) > 0:
            dist_vias = raster_distancia(vias_clip, transform, shape, CRS)
        else:
            dist_vias = np.full(shape, np.nan)
            print(f"  ! Vías no encontradas — usando NaN")
        print(f"  ✓ Distancia vías")

        # ── 5. Distancia a centros poblados ──────────────────
        pob_path = Path(PATH_POBLADOS)
        pob_files = list(pob_path.glob(f"{codigo}_*.gpkg"))
        if pob_files:
            pob = gpd.read_file(pob_files[0]).to_crs(CRS)
            dist_pob = raster_distancia(pob, transform, shape, CRS)
        else:
            dist_pob = np.full(shape, np.nan)
            print(f"  ! Centros poblados no encontrados — usando NaN")
        print(f"  ✓ Distancia centros poblados")

        # ── 6. Distancia a lugares de referencia (EQUITAS) ───
        refs = gdf_dilig[
            (gdf_dilig['Cod_Mun'] == codigo)
        ].copy()
        if len(refs) > 0:
            dist_refs = raster_distancia(refs, transform, shape, CRS)
        else:
            dist_refs = np.full(shape, np.nan)
        print(f"  ✓ Distancia lugares de referencia")

        # ── 7. Variable de resultado (presencias) ────────────
        presencias = gdf_dilig[
        (gdf_dilig['Cod_Mun'] == codigo) & 
        (gdf_dilig['label'] == 1.0) &
        (~gdf_dilig['Tipo disposición (Cuando aplique)'].isin(excluir_tipos))
        ].copy()
        ausencias = gdf_dilig[
            (gdf_dilig['Cod_Mun'] == codigo) & 
            (gdf_dilig['label'] == 0.0)
        ].copy()

        # ── 8. Guardar stack de covariables ──────────────────
        stack = np.stack([
            ndvi_arr,
            dem_arr,
            slope_arr,
            dist_hidro,
            dist_vias,
            dist_pob,
            dist_refs
        ], axis=0)  # shape: (7, height, width)

        output_path = Path(PATH_OUTPUT) / f"covariables_{codigo}.npy"
        np.save(output_path, stack)

        # Guardar también las presencias y ausencias como CSV
        if len(presencias) > 0:
            presencias[['lon', 'lat', 'label']].to_csv(
                Path(PATH_OUTPUT) / f"presencias_{codigo}.csv", index=False
            )
        if len(ausencias) > 0:
            ausencias[['lon', 'lat', 'label']].to_csv(
                Path(PATH_OUTPUT) / f"ausencias_{codigo}.csv", index=False
            )

        resultados.append({
            'codigo': codigo,
            'n_presencias': len(presencias),
            'n_ausencias': len(ausencias),
            'shape': shape,
            'status': 'ok'
        })
        print(f"  ✓ Guardado: {output_path.name}")
        print(f"     Presencias: {len(presencias)}, Ausencias: {len(ausencias)}")

    except Exception as e:
        print(f"  ✗ Error: {e}")
        errores.append((codigo, str(e)))


In [ ]:
# ============================================================
# RESUMEN
# ============================================================
print(f"\n{'='*50}")
print(f"Procesados exitosamente: {len(resultados)}")
print(f"Errores: {len(errores)}")
if errores:
    print("\nMunicipios con error:")
    for cod, err in errores:
        print(f"  {cod}: {err}")

df_resumen = pd.DataFrame(resultados)
if len(df_resumen) > 0:
    print(f"\nResumen:")
    print(df_resumen[['codigo', 'n_presencias', 'n_ausencias', 'shape']].to_string())
    df_resumen.to_csv(Path(PATH_OUTPUT) / "resumen_procesamiento.csv", index=False)

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'elapid'], 
               capture_output=True)
print("✓ elapid instalado")

In [ ]:
import pandas as pd

resumen = pd.read_csv("../data/Processed/resumen_procesamiento.csv")
print("Municipios con presencias >= 5:")
print(resumen[resumen['n_presencias'] >= 5][
    ['codigo', 'n_presencias', 'n_ausencias']
].sort_values('n_presencias', ascending=False).to_string())

print(f"\nTotal presencias disponibles: {resumen['n_presencias'].sum()}")
print(f"Total ausencias disponibles: {resumen['n_ausencias'].sum()}")

## Nuevo procesamiento

In [ ]:
import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd
from scipy.ndimage import distance_transform_edt
from rasterio.features import rasterize
from rasterio.transform import from_bounds
from skimage.transform import resize
from pathlib import Path
import json
import os
import re

# ============================================================
# PATHS
# ============================================================
PATH_MUNICIPIOS  = "../data/Municipios_Diciembre_2025_shp/Munpio.shp"
PATH_NDVI        = "../data/landsat/landsat_ndvi/"
PATH_DEM         = "../data/dem/dem_srtm/"
PATH_HIDRO       = "../data/hidrografia/"
PATH_VIAS_OSM    = "../data/red_vial_osm/"
PATH_POBLADOS    = "../data/centros_poblados/"
PATH_GEOJSON     = "../data/Processed/full_MIG.geojson"
PATH_AÑOS        = "../data/Processed/anios_asignados.csv"
PATH_OUTPUT      = "../data/Processed/"
RESOLUCION       = 30
CRS              = "EPSG:4326"

os.makedirs(PATH_OUTPUT, exist_ok=True)

In [ ]:

# ============================================================
# CARGAR DATOS BASE
# ============================================================
print("Cargando datos base...")

municipios = gpd.read_file(PATH_MUNICIPIOS)
municipios['MpCodigo'] = municipios['MpCodigo'].astype(str).str.zfill(5)
municipios = municipios.to_crs(CRS)

# Cargar geojson con coordenadas corregidas
with open(PATH_GEOJSON) as f:
    data = json.load(f)

props = [f['properties'] for f in data['features']]
coords = [f['geometry']['coordinates'] for f in data['features']]
df_dilig = pd.DataFrame(props)
df_dilig['lon_orig'] = [c[0] for c in coords]
df_dilig['lat_orig'] = [c[1] for c in coords]
df_dilig['Cod_Mun'] = df_dilig['Cod_Mun'].astype(str).str.zfill(5)

# Aplicar corrección de coordenadas

def parsear_coordenada(valor):
    if valor is None or str(valor).strip() in ['', 'SD', 'nan']:
        return None
    valor = str(valor).strip()
    try:
        return float(valor)
    except:
        pass
    patron = r"([NSns]?)\s*(\d+)[°]\s*(\d+)['']\s*([\d,.]+)[\"″]?\s*([EWew]?)"
    match = re.search(patron, valor)
    if match:
        h_pre = match.group(1).upper()
        grados = float(match.group(2))
        minutos = float(match.group(3))
        segundos = float(match.group(4).replace(',', '.'))
        decimal = grados + minutos/60 + segundos/3600
        h_post = match.group(5).upper()
        if (h_pre or h_post) in ['S', 'W']:
            decimal = -decimal
        return decimal
    return None

def corregir_fila(row):
    def es_lat(v): return v is not None and -5 <= v <= 13
    def es_lon(v): return v is not None and -80 <= v <= -66

    lat_orig = parsear_coordenada(row.get('lat'))
    lon_orig = parsear_coordenada(row.get('long'))

    if es_lat(lat_orig) and es_lon(lon_orig):
        return lat_orig, lon_orig
    if es_lon(lat_orig) and es_lat(lon_orig):
        return lon_orig, lat_orig

    try:
        lat_dd = float(row.get('lat_dd', np.nan))
        lon_dd = float(row.get('lon_dd', np.nan))
        if es_lat(lat_dd) and es_lon(lon_dd):
            return lat_dd, lon_dd
        if es_lon(lat_dd) and es_lat(lon_dd):
            return lon_dd, lat_dd
    except:
        pass

    return None, None

lats, lons = [], []
for _, row in df_dilig.iterrows():
    lat, lon = corregir_fila(row)
    lats.append(lat)
    lons.append(lon)

df_dilig['lat'] = lats
df_dilig['lon'] = lons
df_dilig_clean = df_dilig[df_dilig['lat'].notna()].copy()
print(f"1. Después de coordenadas: {len(df_dilig_clean)}")

df_años = pd.read_csv(PATH_AÑOS)
print(f"2. Antes del merge: {len(df_dilig_clean)}")

df_dilig_clean = df_dilig_clean.merge(
    df_años[['COD_REF_EQ', 'año_asignado', 'fuente_año']],
    on='COD_REF_EQ', how='left'
)
print(f"3. Después del merge: {len(df_dilig_clean)}")

df_dilig_clean = df_dilig_clean[
    ~df_dilig_clean['Tipo disposición (Cuando aplique)'].isin(['Cementerio'])
].copy()
print(f"4. Después de excluir cementerios: {len(df_dilig_clean)}")

gdf_dilig = gpd.GeoDataFrame(
    df_dilig_clean,
    geometry=gpd.points_from_xy(df_dilig_clean['lon'], df_dilig_clean['lat']),
    crs=CRS
)
print(f"5. GeoDataFrame final: {len(gdf_dilig)}")
print(f"✓ {len(municipios)} municipios")
print(f"✓ {len(gdf_dilig)} diligencias válidas")


In [ ]:
# ============================================================
# FUNCIÓN DE DISTANCIA
# ============================================================
def raster_distancia(gdf_lineas, transform, shape):
    if len(gdf_lineas) == 0:
        return np.full(shape, np.nan)
    try:
        burned = rasterize(
            [(geom, 1) for geom in gdf_lineas.geometry 
             if geom is not None and not geom.is_empty],
            out_shape=shape,
            transform=transform,
            fill=0,
            dtype='uint8'
        )
        dist = distance_transform_edt(1 - burned) * RESOLUCION
        return dist
    except:
        return np.full(shape, np.nan)

def derivar_pendiente(dem_array):
    dy, dx = np.gradient(dem_array, RESOLUCION)
    return np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))

def cargar_raster(path_carpeta, codigo, prefijo):
    carpeta = Path(path_carpeta)
    # Buscar versión v2 primero (período corregido)
    archivos_v2 = list(carpeta.glob(f"{prefijo}_{codigo}_v2.tif"))
    if archivos_v2:
        path = archivos_v2[0]
    else:
        archivos = list(carpeta.glob(f"{prefijo}_{codigo}.tif"))
        if not archivos:
            return None, None
        path = archivos[0]
    with rasterio.open(path) as src:
        arr = src.read(1).astype(float)
        if src.nodata:
            arr[arr == src.nodata] = np.nan
        return arr, src.transform


In [ ]:
# ============================================================
# PROCESAR CADA MUNICIPIO
# ============================================================
codigos = [c for c in gdf_dilig['Cod_Mun'].unique() if c != '000SD']

resultados = []
errores = []

for codigo in codigos:
    print(f"\nProcesando: {codigo}")
    
    try:
        mun = municipios[municipios['MpCodigo'] == codigo]
        if len(mun) == 0:
            print(f"  ✗ Municipio no encontrado")
            errores.append((codigo, 'municipio no encontrado'))
            continue

        bounds = mun.total_bounds
        width  = max(int((bounds[2]-bounds[0])/(RESOLUCION/111320)), 10)
        height = max(int((bounds[3]-bounds[1])/(RESOLUCION/111320)), 10)
        transform = from_bounds(*bounds, width, height)
        shape = (height, width)

        # ── NDVI ─────────────────────────────────────────────
        ndvi_arr, _ = cargar_raster(PATH_NDVI, codigo, 'ndvi')
        if ndvi_arr is None:
            print(f"  ✗ NDVI no encontrado")
            errores.append((codigo, 'NDVI no encontrado'))
            continue
        ndvi_arr = resize(ndvi_arr, shape, anti_aliasing=True)
        print(f"  ✓ NDVI: {ndvi_arr.shape}")

        # ── DEM → altitud y pendiente ─────────────────────────
        dem_arr, _ = cargar_raster(PATH_DEM, codigo, 'dem')
        if dem_arr is None:
            print(f"  ✗ DEM no encontrado")
            errores.append((codigo, 'DEM no encontrado'))
            continue
        dem_arr   = resize(dem_arr, shape, anti_aliasing=True)
        slope_arr = derivar_pendiente(dem_arr)
        print(f"  ✓ DEM + pendiente")

        # ── Hidrografía ───────────────────────────────────────
        hidro_files = list(Path(PATH_HIDRO).glob(f"{codigo}_*.gpkg"))
        if hidro_files:
            hidro = gpd.read_file(hidro_files[0]).to_crs(CRS)
            dist_hidro = raster_distancia(hidro, transform, shape)
        else:
            dist_hidro = np.full(shape, np.nan)
            print(f"  ! Sin hidrografía")
        print(f"  ✓ Hidrografía")

        # ── Vías OSM ─────────────────────────────────────────
        vias_osm_path = Path(PATH_VIAS_OSM) / f"{codigo}_vias.gpkg"
        if vias_osm_path.exists():
            vias = gpd.read_file(vias_osm_path).to_crs(CRS)
            dist_vias = raster_distancia(vias, transform, shape)
            print(f"  ✓ Vías OSM: {len(vias)} segmentos")
        else:
            dist_vias = np.full(shape, np.nan)
            print(f"  ! Sin vías OSM")

        # ── Centros poblados ──────────────────────────────────
        pob_files = list(Path(PATH_POBLADOS).glob(f"{codigo}_*.gpkg"))
        if pob_files:
            pob = gpd.read_file(pob_files[0]).to_crs(CRS)
            dist_pob = raster_distancia(pob, transform, shape)
        else:
            dist_pob = np.full(shape, np.nan)
            print(f"  ! Sin centros poblados")
        print(f"  ✓ Centros poblados")

        # ── Lugares de referencia EQUITAS ─────────────────────
        refs = gdf_dilig[gdf_dilig['Cod_Mun'] == codigo].copy()
        if len(refs) > 0:
            dist_refs = raster_distancia(refs, transform, shape)
        else:
            dist_refs = np.full(shape, np.nan)
        print(f"  ✓ Referencias EQUITAS")

        # ── Variable de resultado ─────────────────────────────
        excluir = ['Cementerio']
        presencias = gdf_dilig[
            (gdf_dilig['Cod_Mun'] == codigo) &
            (gdf_dilig['label'] == 1.0) &
            (~gdf_dilig['Tipo disposición (Cuando aplique)'].isin(excluir))
        ].copy()

        ausencias = gdf_dilig[
            (gdf_dilig['Cod_Mun'] == codigo) &
            (gdf_dilig['label'] == 0.0)
        ].copy()

        # ── Guardar stack ─────────────────────────────────────
        stack = np.stack([
            ndvi_arr, dem_arr, slope_arr,
            dist_hidro, dist_vias, dist_pob, dist_refs
        ], axis=0)

        np.save(Path(PATH_OUTPUT) / f"covariables_{codigo}.npy", stack)

        if len(presencias) > 0:
            presencias[['lon', 'lat', 'label', 
                        'año_asignado', 'fuente_año']].to_csv(
                Path(PATH_OUTPUT) / f"presencias_{codigo}.csv", index=False
            )
        if len(ausencias) > 0:
            ausencias[['lon', 'lat', 'label',
                       'año_asignado', 'fuente_año']].to_csv(
                Path(PATH_OUTPUT) / f"ausencias_{codigo}.csv", index=False
            )

        resultados.append({
            'codigo': codigo,
            'n_presencias': len(presencias),
            'n_ausencias': len(ausencias),
            'status': 'ok'
        })
        print(f"  ✓ Guardado — pres={len(presencias)}, ause={len(ausencias)}")

    except Exception as e:
        print(f"  ✗ Error: {e}")
        errores.append((codigo, str(e)))



In [ ]:
# ============================================================
# RESUMEN
# ============================================================
print(f"\n{'='*50}")
print(f"Procesados: {len(resultados)}")
print(f"Errores: {len(errores)}")
if errores:
    for cod, err in errores:
        print(f"  {cod}: {err}")

pd.DataFrame(resultados).to_csv(
    Path(PATH_OUTPUT) / "resumen_procesamiento_v2.csv", index=False
)